
# Worldwide Language Pilot — round22 실제 데이터 병행 산출

**이 노트북이 하는 일과 하지 않는 일을 먼저 밝힙니다.**

"Worldwide Language Pilot"은 최종 v7 리포트(10,020건, K=10·M=5·실루엣=0.267)의
7.6절에서만 언급되는 지표이며, 그 원본 서술과 원본 산출 JSON
(`worldwide_language_index_live_reference_v7.json`)은 이번 세션에 없습니다.
남아있는 것은 그 지표를 계산했던 스크립트
(`scripts/indices_csv/build_worldwide_language_index_csv.py`)의 컬럼
스키마뿐입니다.

이 노트북은 그 스키마의 정의를, 이번 세션에 실제로 복구된 v7 라운드22
스냅샷(100개 팬덤, 5,612건, `data/v6_r22_snapshot/fandom_scores_v6.json`)에
적용합니다. 원본 CSV의 재현이 아니라 **독립적인 병행 산출**이며, 두 가지
이유로 절대값이 다릅니다: (1) round22는 10개 언어만 다루고(최종 리포트는
필리핀어·포르투갈어·튀르키예어·아랍어 4개가 더 있었던 것으로 보임),
(2) round22는 5,612건, 최종 리포트는 10,020건 기준입니다.

핵심은 이 노트북 1절에서 **LanguageCoverage 공식 자체가 진짜인지를 실제
데이터로 재검증**한다는 것입니다 — 이 공식이 검증되어야, 그 위에 얹는
"해외 언어" 파일럿 지표(2~3절)도 근거 있는 계산이 됩니다.


In [1]:

import json
import math
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data/v6_r22_snapshot")

with open(DATA_DIR / "fandom_scores_v6.json", encoding="utf-8") as f:
    fandom_scores = json.load(f)

print(f"fandom_scores_v6.json: {len(fandom_scores)}개 팬덤 레코드")

ALL_LANGS = ["ko", "en", "ja", "zh", "es", "fr", "th", "id", "vi", "ru"]

langs_present = set()
for rec in fandom_scores:
    langs_present.update(rec["coverage_detail"]["language_counts"].keys())
print("실제 데이터에 등장하는 언어 집합:", sorted(langs_present))
print("ALL_LANGS(기술 상세명세서 6.1절 기준 10개)와 일치 =", langs_present <= set(ALL_LANGS))


fandom_scores_v6.json: 100개 팬덤 레코드
실제 데이터에 등장하는 언어 집합: ['en', 'es', 'fr', 'id', 'ja', 'ko', 'ru', 'th', 'vi', 'zh']
ALL_LANGS(기술 상세명세서 6.1절 기준 10개)와 일치 = True



## 1. LanguageCoverage 공식 재검증

`docs/LDA_V6_V7_TECHNICAL_SPECIFICATION.md` 6.1절의 공식:

```
LanguageCoverage(f) = -Σ_l p(l)·ln(p(l)) / ln(10)   (l ∈ {ko,en,ja,zh,es,fr,th,id,vi,ru})
```

이 공식을 100개 팬덤 전원에 대해 직접 재계산해, 실제 저장된
`coverage_detail.language_coverage` 값과 정확히 일치하는지 대조합니다. 또한
언어별 언급 수의 합이 `n_evidence`(그 팬덤의 전체 근거 문장 수)와 일치하는지도
함께 확인합니다.


In [2]:

def shannon_diversity(counts: dict, denom_langs: int) -> float:
    total = sum(counts.values())
    if total == 0 or denom_langs <= 1:
        return 0.0
    ent = 0.0
    for c in counts.values():
        if c == 0:
            continue
        p = c / total
        ent += -p * math.log(p)
    return ent / math.log(denom_langs)


rows = []
sum_mismatch = 0
cov_mismatch = 0

for rec in fandom_scores:
    cd = rec["coverage_detail"]
    counts = cd["language_counts"]
    n_evidence = cd["n_evidence"]

    lang_sum = sum(counts.values())
    if lang_sum != n_evidence:
        sum_mismatch += 1

    recomputed_cov = shannon_diversity(counts, len(ALL_LANGS))
    diff = abs(recomputed_cov - cd["language_coverage"])
    if diff > 0.001:
        cov_mismatch += 1

    rows.append({
        "fandom": rec["fandom"],
        "n_evidence": n_evidence,
        "lang_sum": lang_sum,
        "language_coverage_recorded": cd["language_coverage"],
        "language_coverage_recomputed": round(recomputed_cov, 4),
        "diff": round(diff, 4),
    })

check_df = pd.DataFrame(rows)
print(f"언어별 언급 합 != n_evidence 인 팬덤 수: {sum_mismatch} / {len(fandom_scores)}")
print(f"LanguageCoverage 재계산 불일치(오차 >= 0.001) 팬덤 수: {cov_mismatch} / {len(fandom_scores)}")
check_df.sort_values('diff', ascending=False).head(5)


언어별 언급 합 != n_evidence 인 팬덤 수: 0 / 100
LanguageCoverage 재계산 불일치(오차 >= 0.001) 팬덤 수: 0 / 100


,fandom,n_evidence,lang_sum,language_coverage_recorded,language_coverage_recomputed,diff
32,IKON,64,64,0.670,0.6695,0.0005
33,Crush,70,70,0.618,0.6175,0.0005
12,이영지,79,79,0.478,0.4785,0.0005
10,싸이,65,65,0.549,0.5485,0.0005
63,이효리,51,51,0.412,0.4115,0.0005



**결론:** 두 검증 모두 불일치 0건이면, 6.1절 공식이 실제 파이프라인 산출물과
정확히 일치한다는 뜻이고, 이 위에 얹는 "해외 언어" 파일럿 지표(아래)도
같은 원재료(`language_counts`)에서 파생된 근거 있는 계산이 됩니다.



## 2. Worldwide Language Pilot 지표 계산 (원본 스크립트 컬럼 스키마 적용)

원본 `build_worldwide_language_index_csv.py`가 정의한 컬럼을 그대로
계산합니다. "해외언어다양성"의 정규화 분모(9 = 한국어를 제외한 9개 언어)는
원본에 명시되어 있지 않아 이 노트북이 추정한 값임을 다시 한 번 밝힙니다.


In [3]:

LANG_LABEL = {
    "ko": "한국어", "en": "영어", "ja": "일본어", "zh": "중국어", "es": "스페인어",
    "fr": "프랑스어", "th": "태국어", "id": "인도네시아어", "vi": "베트남어", "ru": "러시아어",
}
FOREIGN_LANGS = [l for l in ALL_LANGS if l != "ko"]

pilot_rows = []
for rec in fandom_scores:
    cd = rec["coverage_detail"]
    counts = {l: cd["language_counts"].get(l, 0) for l in ALL_LANGS}
    total = sum(counts.values())
    foreign_counts = {l: counts[l] for l in FOREIGN_LANGS}
    foreign_total = sum(foreign_counts.values())

    n_languages_hit = sum(1 for c in counts.values() if c > 0)
    n_foreign_hit = sum(1 for c in foreign_counts.values() if c > 0)

    if foreign_total > 0:
        primary_lang = max(foreign_counts, key=foreign_counts.get)
        primary_share = foreign_counts[primary_lang] / foreign_total
    else:
        primary_lang, primary_share = None, 0.0

    row = {
        "팬덤": rec["fandom"],
        "근거문장수": total,
        "총언어언급": total,
        "검출언어수": n_languages_hit,
        "언어다양성": round(shannon_diversity(counts, len(ALL_LANGS)), 4),
        "해외근거문장수": foreign_total,
        "해외비중": round(foreign_total / total, 4) if total else 0.0,
        "검출해외언어수": n_foreign_hit,
        "해외언어다양성": round(shannon_diversity(foreign_counts, len(FOREIGN_LANGS)), 4),
        "대표해외언어": LANG_LABEL.get(primary_lang, "") if primary_lang else "",
        "대표해외언어비중": round(primary_share, 4),
    }
    for l in ALL_LANGS:
        row[LANG_LABEL[l]] = counts[l]
    pilot_rows.append(row)

pilot_df = pd.DataFrame(pilot_rows).sort_values(["해외근거문장수", "팬덤"], ascending=[False, True]).reset_index(drop=True)
print(f"산출된 행 수: {len(pilot_df)}")
pilot_df.head(15)


산출된 행 수: 100


,팬덤,근거문장수,총언어언급,검출언어수,언어다양성,해외근거문장수,해외비중,검출해외언어수,해외언어다양성,대표해외언어,...,한국어,영어,일본어,중국어,스페인어,프랑스어,태국어,인도네시아어,베트남어,러시아어
0,BLACKPINK,123,123,9,0.8293,95,0.7724,8,0.8091,영어,...,28,39,9,11,13,0,8,4,7,4
1,Stray Kids,111,111,8,0.7182,85,0.7658,7,0.6593,영어,...,26,46,9,1,8,0,4,8,0,9
2,NewJeans,100,100,9,0.6401,82,0.8200,8,0.5564,영어,...,18,54,1,8,6,3,2,0,7,1
3,BTS,125,125,8,0.6832,81,0.6480,7,0.6493,영어,...,44,42,14,4,12,0,0,4,1,4
4,SEVENTEEN,113,113,8,0.5476,81,0.7168,7,0.4222,영어,...,32,62,5,5,4,0,1,3,1,0
5,LE SSERAFIM,91,91,8,0.7310,66,0.7253,7,0.6874,영어,...,25,30,16,2,7,0,2,6,0,3
6,TWICE,111,111,8,0.6848,66,0.5946,7,0.6902,영어,...,45,32,12,6,8,0,4,2,0,2
7,NCT,90,90,9,0.7083,65,0.7222,8,0.6555,영어,...,25,37,8,7,2,0,4,4,2,1
8,레드벨벳,69,69,8,0.7588,52,0.7536,7,0.7180,영어,...,17,23,5,1,5,0,11,4,3,0
9,ITZY,61,61,8,0.6743,50,0.8197,7,0.6001,영어,...,11,30,6,1,4,0,0,5,1,3



## 3. 자체 검증 — 무결성 재확인 (원본 스크립트와 동일한 검증 패턴)

원본 스크립트가 자체적으로 수행하던 두 가지 검증(언어별 합계가
`total_group_bullets`와 일치하는지, `foreign_bullets`가 ko를 제외한 합과
일치하는지)을 이 병행 산출물에도 동일하게 적용합니다.


In [4]:

mismatches_total = 0
mismatches_foreign = 0

for rec, prow in zip(fandom_scores, pilot_rows):
    cd = rec["coverage_detail"]
    counts = {l: cd["language_counts"].get(l, 0) for l in ALL_LANGS}
    s_all = sum(counts.values())
    s_foreign = sum(v for k, v in counts.items() if k != "ko")
    if s_all != prow["근거문장수"]:
        mismatches_total += 1
    if s_foreign != prow["해외근거문장수"]:
        mismatches_foreign += 1

print(f"mismatches_total = {mismatches_total} / {len(fandom_scores)}")
print(f"mismatches_foreign = {mismatches_foreign} / {len(fandom_scores)}")
print(f"해외근거문장수 상위 10개 팬덤: {list(pilot_df['팬덤'].head(10))}")


mismatches_total = 0 / 100
mismatches_foreign = 0 / 100
해외근거문장수 상위 10개 팬덤: ['BLACKPINK', 'Stray Kids', 'NewJeans', 'BTS', 'SEVENTEEN', 'LE SSERAFIM', 'TWICE', 'NCT', '레드벨벳', 'ITZY']



## 4. 요약 — 코퍼스 전체 언어 구성

round22 스냅샷(100개 팬덤, 5,612건) 전체를 기준으로 언어별 총 언급 수와
비중을 집계합니다. 이 값은 이전에 정리한
`docs/TOKENIZER_WORDCLOUD_REPORT.md`가 인용한 최종 v7 리포트의 언어 분포
(10,020건 기준)와는 코퍼스 크기가 다르므로 절대값이 아니라 방향성만
비교해야 합니다.


In [5]:

total_by_lang = {LANG_LABEL[l]: sum(r[LANG_LABEL[l]] for r in pilot_rows) for l in ALL_LANGS}
grand_total = sum(total_by_lang.values())

summary_df = pd.DataFrame([
    {"언어": lang, "총언급수": cnt, "비중": round(cnt / grand_total, 4)}
    for lang, cnt in sorted(total_by_lang.items(), key=lambda x: -x[1])
])
print(f"round22 전체 언어 언급 총계: {grand_total}건 (100개 팬덤, 5,612건 근거문장 기준)")
summary_df


round22 전체 언어 언급 총계: 5612건 (100개 팬덤, 5,612건 근거문장 기준)


,언어,총언급수,비중
0,한국어,2956,0.5267
1,영어,1561,0.2782
2,일본어,304,0.0542
3,인도네시아어,192,0.0342
4,태국어,181,0.0323
5,중국어,152,0.0271
6,스페인어,121,0.0216
7,베트남어,72,0.0128
8,러시아어,69,0.0123
9,프랑스어,4,0.0007



## 5. 한계

1. round22는 10개 언어만 다루며, 최종 v7 리포트의 14개 언어(필리핀어·
   포르투갈어·튀르키예어·아랍어 미포함) 체계를 재현하지 않습니다.
2. "해외언어다양성"의 정규화 분모(`ln(9)`)는 원본 스크립트·문서에 명시된
   값이 아니라 이 노트북이 "가능한 9개 해외 언어" 가정 하에 추정한
   것입니다.
3. 이 표의 절대 수치(근거문장수·해외비중 등)는 round22(5,612건) 기준이며,
   최종 v7 리포트(10,020건) 기준 `worldwide_language_index_v7.csv`와
   직접 비교할 수 없습니다 — 코퍼스 크기 자체가 다릅니다.
4. 원본 "Worldwide Language Pilot"이 실제로 무엇을 결론으로 제시했는지
   (예: 특정 팬덤의 해외 확산 패턴에 대한 해석)는 원본 문서가 없어 알 수
   없습니다. 이 노트북은 순수하게 지표 계산만 재현합니다.
